# 08 — Agents, MCP Tools & A2A Protocol Demo\n\nDemonstrates the agent/tool orchestration layer end to end, using the real production modules — nothing here is a notebook-local reimplementation:\n\n1. **MCP tools** (`mcp/tools/*.py`) — called directly, showing exactly what they return today, honestly including the ones that are still documented stubs (with TODOs pointing at the real data source), alongside the equivalent real data pulled through `api/services/*.py` (the sibling REST adapter around the same domain core, per ADR-014).\n2. **Human-in-the-loop approval queue** (`agents/recommendation/approval_queue.py`) — the concrete state-machine mechanism behind ADR-006 ("no agent auto-executes a consequential action"). Enqueue a real item, show it PENDING, approve it, and show the guard that makes it structurally impossible to reach EXECUTED without going through `approve()` first.\n3. **A2A (Agent2Agent) protocol** (`agents/a2a/`) — boots the real FastAPI A2A server as a subprocess, queries agent discovery (`/.well-known/agent.json`) and `tasks/send` for all four wrapped agents (orchestrator/quality/recommendation/monitoring), captures the real JSON responses, and then **cleanly stops the server** at the end — no orphaned process left running.\n\n**Honest LLM-gateway caveat** (flagged again inline where it matters): `agents/llm_gateway/router.py`'s provider adapters are intentional stubs in this environment (no API key configured — see `BUILD_LOG.md`). None of the agents below invoke that gateway for reasoning; the orchestrator agent's intent detection is deliberately **rule-based/heuristic** (keyword routing over a closed `Intent` enum, see `agents/orchestrator/customer_intelligence_agent.py::detect_intent`), which is exactly why it works here with no LLM key. Nothing in this notebook fabricates an LLM response."

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# A real, existing master_customer_id from the MDM Golden Record output (data/mdm/golden_record.parquet)
DEMO_CUSTOMER_ID = "MC00000000"

print(f"ROOT resolved to: {ROOT}")
print(f"Demo customer id: {DEMO_CUSTOMER_ID}")

ROOT resolved to: G:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\Enterprise Customer Intelligence Data Platform (Terminar)
Demo customer id: MC00000000


## 1. MCP tools — real calls, real (including honestly-empty) data\n\n`mcp/tools/customer.py::get_customer()` and `mcp/tools/quality.py::get_data_quality()` are built with their **final, agreed contract** already (ROADMAP.md Sprint 13), but the data-access wiring is a documented `TODO` in each — they currently return `found=False` / `None` scores by design, not by bug. This is called out explicitly in each module's docstring, and confirmed live below.\n\nRight next to each stub call, the same lookup is repeated through `api/services/*.py` — the sibling REST adapter around the *same* domain core (ADR-014, Ports & Adapters) that reads the actual `data/mdm/`, `data/ml/` parquet outputs and `data_quality/reports/dq_report.json` produced by the earlier work-packages. Same question, two adapters, one currently wired and one not — an honest snapshot of where this sprint's wiring stands.

In [2]:
import json

from mcp.tools.customer import get_customer
from api.services.customer_service import get_golden_record

mcp_customer_result = get_customer(DEMO_CUSTOMER_ID)
api_customer_result = get_golden_record(DEMO_CUSTOMER_ID)

print("mcp/tools/customer.py::get_customer() — documented stub, not yet wired:")
print(json.dumps(mcp_customer_result, indent=2))

print("\napi/services/customer_service.py::get_golden_record() — same question, real data:")
print(json.dumps(api_customer_result, indent=2, default=str))

mcp/tools/customer.py::get_customer() — documented stub, not yet wired:
{
  "master_customer_id": "MC00000000",
  "name": null,
  "email": null,
  "phone": null,
  "address": null,
  "segment": null,
  "source_record_count": 0,
  "found": false
}

api/services/customer_service.py::get_golden_record() — same question, real data:
{
  "master_customer_id": "MC00000000",
  "canonical_name": "Brenda Alves",
  "canonical_email": "samuel32@example.net",
  "canonical_phone": "555181960013",
  "city": "Curitiba",
  "state": "PR",
  "source_record_count": 1,
  "created_at": "2023-12-11 14:19:27",
  "updated_at": "2025-11-03 22:21:57",
  "found": true
}


In [3]:
from mcp.tools.quality import get_data_quality

mcp_dq_result = get_data_quality()
print("mcp/tools/quality.py::get_data_quality() — documented stub, not yet wired:")
print(json.dumps(mcp_dq_result, indent=2))

# The real report this tool's TODO says it should read from — data_quality/ is a read-only,
# already-completed module for this notebook (WP1), so this is the actual GX-style rollup.
dq_report_path = ROOT / "data_quality" / "reports" / "dq_report.json"
dq_report = json.loads(dq_report_path.read_text(encoding="utf-8"))
print("\ndata_quality/reports/dq_report.json — the real platform DQ rollup this tool should read:")
print(json.dumps({"overall_score_pct": dq_report["overall_score_pct"], "generated_at": dq_report["generated_at"],
                   "tables": [{"table_name": t["table_name"], "score": t["score"], "quarantined_count": t["quarantined_count"]}
                              for t in dq_report["tables"]]}, indent=2))

mcp/tools/quality.py::get_data_quality() — documented stub, not yet wired:
{
  "dataset": null,
  "dq_score": null,
  "dimension_scores": {
    "completeness": null,
    "validity": null,
    "uniqueness": null,
    "referential_integrity": null,
    "freshness": null,
    "schema": null,
    "volume_anomaly": null
  },
  "as_of": null
}

data_quality/reports/dq_report.json — the real platform DQ rollup this tool should read:
{
  "overall_score_pct": 99.83,
  "generated_at": "2026-08-14T02:37:24.734082+00:00",
  "tables": [
    {
      "table_name": "crm_customer",
      "score": 0.9967,
      "quarantined_count": 116
    },
    {
      "table_name": "support_ticket",
      "score": 0.9986,
      "quarantined_count": 28
    },
    {
      "table_name": "web_event",
      "score": 0.9987,
      "quarantined_count": 571
    },
    {
      "table_name": "campaign_interaction",
      "score": 0.9987,
      "quarantined_count": 228
    },
    {
      "table_name": "payment_finance",
      "

## 2. Human-in-the-loop approval queue (ADR-006)\n\nFirst, the real current agent entrypoint: `agents/recommendation/recommendation_agent.py::generate_recommendation()`. Its context-gathering step (`gather_customer_context`) is itself a documented TODO (same honest-stub pattern as the MCP tools above), so today it always enqueues a `no_action_recommended` / confidence `0.0` item — but that item goes through the **real** `ApprovalQueue` state machine, which is the actual point of this section.\n\nTo exercise every transition of that state machine (not just the placeholder path), the cells below also call `enqueue_recommendation()` directly — the same function the agent calls — with an illustrative, clearly-labeled example payload, and then walk through the full PENDING → APPROVED → EXECUTED lifecycle plus the guard that blocks invalid transitions.

In [4]:
from agents.recommendation.recommendation_agent import generate_recommendation

agent_item = generate_recommendation(DEMO_CUSTOMER_ID)
print("agents/recommendation/recommendation_agent.py::generate_recommendation() — real call, honest current output:")
print(f"  queue_id={agent_item.queue_id}")
print(f"  status={agent_item.status.value}")
print(f"  recommendation={agent_item.recommendation!r}  confidence={agent_item.confidence}")
print(f"  evidence={agent_item.evidence}")

agents/recommendation/recommendation_agent.py::generate_recommendation() — real call, honest current output:
  queue_id=3ec254f5-fc97-4e3c-b4f0-37b35f01a4de
  status=PENDING
  recommendation='no_action_recommended'  confidence=0.0
  evidence=['churn score unavailable — see mcp/tools/ml.py TODO']


In [5]:
from agents.recommendation.approval_queue import ApprovalQueueError, get_queue, enqueue_recommendation

queue = get_queue()  # the same shared singleton generate_recommendation() just used above

# Illustrative example payload (evidence text written by us for the demo, NOT an LLM's output) —
# enqueued through the exact same function the agent calls, to exercise the full state machine.
demo_queue_id = enqueue_recommendation(
    master_customer_id=DEMO_CUSTOMER_ID,
    recommendation="offer_retention_discount",
    confidence=0.87,
    evidence=[
        "illustrative demo payload — churn_probability=0.91 (high risk band)",
        "illustrative demo payload — 2 unresolved support tickets in last 90 days",
    ],
)
item = queue.get(demo_queue_id)
print(f"Enqueued: queue_id={demo_queue_id}  status={item.status.value}")

pending = queue.list_pending()
print(f"\nPENDING items in queue: {len(pending)} (includes both items enqueued above)")

# Approve it — the ONLY human-triggered transition that makes an item eligible for execution.
approved = queue.approve(demo_queue_id, approved_by="yuri.dubbern@gmail.com", reason="Demo approval — notebook 08")
print(f"\nAfter approve(): status={approved.status.value}  decided_by={approved.decided_by}")

executed = queue.mark_executed(demo_queue_id)
print(f"After mark_executed(): status={executed.status.value}  executed_at={executed.executed_at}")

# Guard demo: mark_executed() on a PENDING item must be structurally impossible.
try:
    queue.mark_executed(agent_item.queue_id)
except ApprovalQueueError as exc:
    print(f"\nGuard confirmed on the still-PENDING agent item ({agent_item.queue_id}):")
    print(f"  ApprovalQueueError: {exc}")

# Reject path on that same still-PENDING item, for completeness.
rejected = queue.reject(agent_item.queue_id, rejected_by="yuri.dubbern@gmail.com", reason="Demo rejection — placeholder recommendation, no real signal yet")
print(f"\nAfter reject(): status={rejected.status.value}")

Enqueued: queue_id=6b74227c-464e-4109-a68e-5e994c619396  status=PENDING

PENDING items in queue: 2 (includes both items enqueued above)

After approve(): status=APPROVED  decided_by=yuri.dubbern@gmail.com
After mark_executed(): status=EXECUTED  executed_at=2026-08-22 19:55:59.794884+00:00

Guard confirmed on the still-PENDING agent item (3ec254f5-fc97-4e3c-b4f0-37b35f01a4de):
  ApprovalQueueError: Refusing to execute item 3ec254f5-fc97-4e3c-b4f0-37b35f01a4de in status PENDING: only items that have passed through approve() may be marked EXECUTED. This guard is the concrete mechanism behind ADR-006 — do not bypass it.

After reject(): status=REJECTED


## 3. A2A protocol — boot the real server, call it live, stop it cleanly\n\n`agents/a2a/server.py` is a FastAPI app that wraps (never reimplements) the four existing agents, exposing standard A2A discovery (`GET /agents/{id}/.well-known/agent.json`) and task-send (`POST /agents/{id}/tasks/send`) endpoints — see `agents/a2a/README.md` for the full design notes and the prior live-verification run this reproduces.\n\nThe server is started as a background subprocess (`uvicorn`), polled until it responds, exercised for real over HTTP via the bundled stdlib-only `A2AClient`, and then **terminated in a `finally` block** so the process cannot leak past this notebook even if a cell above raises.

In [6]:
import subprocess
import tempfile
import time
import urllib.error
import urllib.request

A2A_PORT = 8020
A2A_BASE_URL = f"http://127.0.0.1:{A2A_PORT}"

log_file = tempfile.NamedTemporaryFile(prefix="a2a_server_", suffix=".log", delete=False)
print(f"Server log: {log_file.name}")

server_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "agents.a2a.server:app", "--port", str(A2A_PORT), "--log-level", "warning"],
    cwd=str(ROOT),
    stdout=log_file,
    stderr=subprocess.STDOUT,
)
print(f"Started uvicorn subprocess, pid={server_process.pid}")

# Poll until the server responds (or the process died trying to start). If it never comes up,
# kill it before raising — never leave a half-started process behind even on failure here.
deadline = time.time() + 30
server_ready = False
try:
    while time.time() < deadline:
        if server_process.poll() is not None:
            raise RuntimeError(f"Server process exited early with code {server_process.returncode}; see {log_file.name}")
        try:
            with urllib.request.urlopen(f"{A2A_BASE_URL}/agents", timeout=1) as resp:
                if resp.status == 200:
                    server_ready = True
                    break
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            time.sleep(0.5)
    if not server_ready:
        raise RuntimeError(f"Server did not become ready within 30s; see {log_file.name}")
except Exception:
    server_process.terminate()
    server_process.wait(timeout=10)
    raise

print(f"Server ready: {server_ready}")

Server log: C:\Users\Yuri_\AppData\Local\Temp\a2a_server_50y3t5cx.log


Started uvicorn subprocess, pid=21552


Server ready: True


In [7]:
from agents.a2a.client import A2AClient

client = A2AClient(base_url=A2A_BASE_URL)

try:
    agents_index = client.list_agents()
    print("GET /agents:")
    print(json.dumps(agents_index, indent=2))

    print("\nGET /agents/quality/.well-known/agent.json:")
    quality_card = client.discover("quality")
    print(json.dumps(quality_card, indent=2)[:1200] + " ...")

    task_inputs = {
        "orchestrator": {"text": "What is the churn rate this quarter?"},  # METRIC_LOOKUP intent
        "quality": {"text": "silver.crm_customer"},
        "recommendation": {"text": DEMO_CUSTOMER_ID},
        "monitoring": {"text": ""},
    }
    a2a_results = {}
    for agent_id, kwargs in task_inputs.items():
        result = client.send_task(agent_id, **kwargs)
        a2a_results[agent_id] = result
        print(f"\nPOST /agents/{agent_id}/tasks/send -> status={result['status']['state']}")
        print(json.dumps(result, indent=2)[:1500])
finally:
    server_process.terminate()
    try:
        server_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        server_process.kill()
        server_process.wait(timeout=10)
    print(f"\nServer stopped. Exit code: {server_process.returncode}")

GET /agents:
[
  {
    "id": "orchestrator",
    "name": "Customer Intelligence Agent",
    "description": "Answers natural-language business questions by routing to Power BI MCP, Databricks/Cortex MCP, or this platform's own Golden Record + Graph MCP tools, depending on detected intent (metric lookup / deep analysis / identity question / policy question). Wraps agents/orchestrator/customer_intelligence_agent.py \u2014 the compiled LangGraph graph is the actual handler; this card never reimplements its logic."
  },
  {
    "id": "quality",
    "name": "Data Quality Agent",
    "description": "Diagnoses a data quality score drop for a given dataset: main cause (worst-scoring DQ dimension), affected row count, and a recommended action \u2014 never triggers a pipeline re-run itself. Wraps agents/quality/data_quality_agent.py:diagnose_quality_drop()."
  },
  {
    "id": "recommendation",
    "name": "Recommendation Agent",
    "description": "Combines churn score, CLV, support history and 

In [8]:
# Confirm nothing was left running: a follow-up request to the now-stopped server must fail.
try:
    urllib.request.urlopen(f"{A2A_BASE_URL}/agents", timeout=2)
    print("UNEXPECTED: server still responding — process was not actually stopped.")
except (urllib.error.URLError, ConnectionError) as exc:
    print(f"Confirmed stopped — follow-up request failed as expected: {type(exc).__name__}: {exc}")

Confirmed stopped — follow-up request failed as expected: URLError: <urlopen error timed out>


## Maps to the real platform / what needs a real LLM key\n\n- `mcp/tools/*.py` — MCP tool contracts, wired for `customer`/`quality` at the API-adapter level (`api/services/`) but not yet at the MCP-transport level itself; both share the same underlying `data/` artifacts once wired, per ADR-014.\n- `agents/recommendation/approval_queue.py` — ADR-006's human-in-the-loop guarantee, proven above as a structural (not just conventional) guard.\n- `agents/a2a/` — the third protocol adapter (alongside MCP and REST) around the same four agents, proven live with a real subprocess boot/query/teardown cycle.\n- **What needs a real LLM key**: `agents/orchestrator/customer_intelligence_agent.py::reason()` currently returns an empty `answer` — the step where it would call `agents/llm_gateway/router.py` to turn retrieved evidence into a natural-language response is a stub in this environment (no `AZURE_OPENAI_API_KEY`/`OPENAI_API_KEY`/etc. configured — see `BUILD_LOG.md`). Everything demonstrated above it — intent detection, tool routing, MCP/A2A transport, the approval queue — works today with zero LLM calls, and that mechanical layer is exactly what this notebook proves is real.